# Reto 1: Consulta y análisis de datos meteorológicos

## 13. Organiza y prueba tu programa

Dividiremos el programa en funciones con responsabilidades claras y comprobaremos su funcionamiento con datos conocidos. Las pruebas locales permiten verificar los cálculos sin depender de la conexión con la API.

## Organización de los archivos

Todos los ficheros del ejercicio están dentro de la carpeta `13_OrganizaPruebaPrograma`:

```text
13_OrganizaPruebaPrograma/
├── 13_OrganizaPruebaPrograma.ipynb
├── meteorologia.py
└── test_meteorologia.py
```

- `meteorologia.py` contiene las funciones y el programa principal.
- `test_meteorologia.py` contiene las pruebas automáticas.
- Este notebook explica y ejecuta ejemplos de uso.

## Funciones del programa

Cada función realiza una tarea concreta:

- `obtener_datos()`: realiza la petición a Open-Meteo.
- `extraer_series()`: valida la respuesta y extrae las listas necesarias.
- `calcular_media()`: calcula la media de una colección.
- `construir_registros()`: relaciona las fechas con las temperaturas.
- `resumir_periodo()`: calcula el resumen estadístico.
- `ejecutar_programa()`: coordina las funciones anteriores.

Separar estas responsabilidades facilita comprender, reutilizar y probar el código.

In [ ]:
import sys
from pathlib import Path

directorio_ejercicio = (
    Path("Reto1/13_OrganizaPruebaPrograma")
    if Path("Reto1").exists()
    else Path(".")
)

ruta_modulo = str(directorio_ejercicio.resolve())
if ruta_modulo not in sys.path:
    sys.path.insert(0, ruta_modulo)

from meteorologia import (
    calcular_media,
    construir_registros,
    ejecutar_programa,
    extraer_series,
    resumir_periodo,
)

print("Módulo cargado desde:", directorio_ejercicio.resolve())

## 1. Prueba `calcular_media()`

Probamos una lista conocida y los tres casos límite indicados en la guía: una lista vacía, un único valor y temperaturas negativas.

In [ ]:
casos_media = {
    "lista conocida": [4, 6, 8],
    "lista vacía": [],
    "un único valor": [7.5],
    "temperaturas negativas": [-6, -3, 0],
}

for nombre, valores in casos_media.items():
    print(nombre, "->", calcular_media(valores))

### Resultado

- La media de `[4, 6, 8]` es `6`.
- Una lista vacía devuelve `None`, porque no existe una media que calcular.
- La media de un único valor es el propio valor: `7.5`.
- Las temperaturas negativas se calculan normalmente: la media de `[-6, -3, 0]` es `-3`.

## 2. Prueba la construcción de registros

Utilizamos datos pequeños cuyo resultado puede comprobarse manualmente. La función añade también la amplitud térmica diaria.

In [ ]:
fechas_prueba = ["2025-01-01", "2025-01-02"]
medias_prueba = [7.1, 7.3]
maximas_prueba = [12.8, 12.9]
minimas_prueba = [2.4, 2.5]

registros_prueba = construir_registros(
    fechas_prueba,
    medias_prueba,
    maximas_prueba,
    minimas_prueba,
)
resumen_prueba = resumir_periodo(registros_prueba)

print("Registros:")
for registro in registros_prueba:
    print(registro)

print("Resumen:", resumen_prueba)

## 3. Prueba datos incorrectos

Las listas deben tener la misma longitud. Si no coinciden, la función genera un `ValueError` con un mensaje comprensible.

In [ ]:
try:
    construir_registros(
        ["2025-01-01"],
        [7.1, 7.3],
        [12.8],
        [2.4],
    )
except ValueError as error:
    print("Error controlado:", error)

## 4. Ejecuta las pruebas automáticas

El fichero `test_meteorologia.py` utiliza `unittest`, incluido en Python. Sus pruebas trabajan con datos locales, por lo que pueden ejecutarse aunque Open-Meteo no esté disponible.

In [ ]:
import unittest

cargador = unittest.TestLoader()
suite = cargador.discover(
    start_dir=str(directorio_ejercicio),
    pattern="test_meteorologia.py",
)
resultado_pruebas = unittest.TextTestRunner(verbosity=2).run(suite)

print("Pruebas ejecutadas:", resultado_pruebas.testsRun)
print("Resultado correcto:", resultado_pruebas.wasSuccessful())

## 5. Ejecuta el programa con datos reales

Después de probar las funciones por separado, ejecutamos el programa completo. Esta celda sí necesita conexión a Internet y la biblioteca `requests`.

In [ ]:
try:
    registros, resumen = ejecutar_programa()
except Exception as error:
    print("No ha sido posible completar la consulta:", error)

## Comprueba tu comprensión

### 1. ¿Qué ventaja tiene dividir el programa en funciones?

**Respuesta:**

Cada parte tiene una responsabilidad concreta. Esto hace que el programa sea más fácil de leer, reutilizar, corregir y probar. Si cambia la forma de calcular el resumen, por ejemplo, puede modificarse esa función sin reescribir la petición a la API.

### 2. ¿Por qué se utilizan datos conocidos en las pruebas?

**Respuesta:**

Porque conocemos de antemano el resultado correcto y podemos detectar si una función se comporta de forma inesperada. Además, estas pruebas son rápidas y no dependen de cambios o fallos en un servicio externo.

### 3. ¿Por qué deben probarse casos límite?

**Respuesta:**

Los casos límite revelan errores que pueden no aparecer con datos habituales. Una lista vacía podría provocar una división entre cero; un único valor comprueba el caso mínimo válido; y las temperaturas negativas verifican que los cálculos no asumen que todos los valores son positivos.

## Conclusión

El programa se ha separado en funciones independientes y se han creado pruebas para los cálculos, la estructura de los datos y los casos límite. Las pruebas automáticas permiten modificar el programa con mayor seguridad: si un cambio altera un comportamiento esperado, el fichero de pruebas lo señala inmediatamente.